# 17. 蒙特卡洛期权定价

## 学习目标

通过本次学习，你将能够：

1. **理解蒙特卡洛方法的基本原理**
2. **模拟资产价格的随机路径**
3. **用 MC 方法定价欧式期权**，并与 BS 解析解对比
4. **掌握方差缩减技术**（对偶变量法、控制变量法）
5. **理解收敛性**：标准误差与 $\sqrt{n}$ 的关系

## 知识地图

```
蒙特卡洛期权定价
├── 基本原理
│   ├── 大数定律
│   ├── 风险中性定价
│   └── 期望收益的现值
├── 随机路径生成
│   ├── 几何布朗运动 (GBM)
│   ├── 离散化：欧拉方法
│   └── 路径模拟
├── 期权定价
│   ├── 欧式 Call/Put
│   ├── 与 BS 解析解对比
│   └── 置信区间
├── 方差缩减
│   ├── 对偶变量法 (Antithetic Variates)
│   ├── 控制变量法 (Control Variates)
│   └── 方差缩减效果对比
└── 收敛性分析
    ├── 标准误差 ∝ 1/√n
    ├── 收敛曲线
    └── 计算成本 vs 精度权衡
```

## 环境依赖

```bash
pip install numpy scipy matplotlib
```

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子以保证可复现
np.random.seed(42)

%matplotlib inline

---
## 1. 理论基础：蒙特卡洛方法

### 1.1 什么是蒙特卡洛方法？

蒙特卡洛方法的核心思想：**用随机模拟来近似计算期望值**。

对于期权定价，我们想计算：

$$C = e^{-rT} \cdot \mathbb{E}[\max(S_T - K, 0)]$$

其中 $S_T$ 是到期时的标的价格，$K$ 是行权价。

蒙特卡洛方法的做法：
1. 模拟 N 条资产价格路径
2. 计算每条路径的期权 payoff
3. 取平均值作为期望的估计
4. 折现得到期权价格

### 1.2 大数定律保证

根据大数定律，当 $N \to \infty$ 时：

$$\hat{C}_N = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K, 0) \xrightarrow{P} C$$

### 1.3 标准误差

MC 估计的标准误差为：

$$SE = \frac{\sigma_{payoff}}{\sqrt{N}}$$

这意味着：
- **精度提高 10 倍，需要 100 倍的模拟次数**
- MC 方法的收敛速度是 $O(1/\sqrt{N})$，比确定性方法慢

---
## 2. 随机路径生成：几何布朗运动

### 2.1 几何布朗运动 (GBM)

资产价格通常假设服从几何布朗运动：

$$dS = \mu S \, dt + \sigma S \, dW$$

其中：
- $\mu$：漂移率（期望收益率）
- $\sigma$：波动率
- $dW$：维纳过程增量，$dW \sim \sqrt{dt} \cdot N(0,1)$

### 2.2 离散化：欧拉方法

将连续时间离散化，分成 $M$ 步，每步 $\Delta t = T/M$：

$$S_{t+\Delta t} = S_t \cdot \exp\left((\mu - \frac{\sigma^2}{2})\Delta t + \sigma\sqrt{\Delta t} \cdot Z\right)$$

其中 $Z \sim N(0,1)$。

### 2.3 风险中性定价

在风险中性世界中，$\mu = r$（无风险利率），所以：

$$S_{t+\Delta t} = S_t \cdot \exp\left((r - \frac{\sigma^2}{2})\Delta t + \sigma\sqrt{\Delta t} \cdot Z\right)$$

In [ ]:
def simulate_gbm_paths(S0, r, sigma, T, M, N):
    """
    模拟几何布朗运动路径
    
    Parameters
    ----------
    S0 : float - 初始价格
    r : float - 无风险利率
    sigma : float - 波动率
    T : float - 到期时间（年）
    M : int - 时间步数
    N : int - 路径数量
    
    Returns
    -------
    np.ndarray : 形状为 (N, M+1) 的路径矩阵
    """
    dt = T / M
    
    # 生成随机数 Z ~ N(0, 1)
    Z = np.random.standard_normal((N, M))
    
    # 初始化路径矩阵
    paths = np.zeros((N, M + 1))
    paths[:, 0] = S0
    
    # 模拟路径
    for t in range(M):
        paths[:, t + 1] = paths[:, t] * np.exp(
            (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t]
        )
    
    return paths


# 模拟参数
S0 = 100      # 初始价格
r = 0.05      # 无风险利率
sigma = 0.2   # 波动率
T = 1.0       # 到期时间 1 年
M = 252       # 252 个交易日
N = 1000      # 模拟 1000 条路径

# 模拟路径
paths = simulate_gbm_paths(S0, r, sigma, T, M, N)

print(f"模拟参数：")
print(f"  初始价格 S0 = {S0}")
print(f"  无风险利率 r = {r}")
print(f"  波动率 σ = {sigma}")
print(f"  到期时间 T = {T} 年")
print(f"  时间步数 M = {M}")
print(f"  路径数量 N = {N}")
print(f"\n路径矩阵形状: {paths.shape}")
print(f"到期价格统计：")
print(f"  均值: {paths[:, -1].mean():.2f}")
print(f"  标准差: {paths[:, -1].std():.2f}")
print(f"  最小值: {paths[:, -1].min():.2f}")
print(f"  最大值: {paths[:, -1].max():.2f}")

In [ ]:
# 绘制模拟路径
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：部分路径
t_axis = np.linspace(0, T, M + 1)
for i in range(min(50, N)):  # 只画前 50 条路径
    axes[0].plot(t_axis, paths[i], alpha=0.1, linewidth=0.5)

# 画均值路径
mean_path = paths.mean(axis=0)
axes[0].plot(t_axis, mean_path, 'b-', linewidth=2, label='均值路径')
axes[0].axhline(y=S0, color='gray', linestyle=':', alpha=0.5, label=f'初始价格 S0={S0}')

axes[0].set_xlabel('时间 (年)', fontsize=12)
axes[0].set_ylabel('标的价格', fontsize=12)
axes[0].set_title('GBM 模拟路径（前 50 条）', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右图：到期价格分布
final_prices = paths[:, -1]
axes[1].hist(final_prices, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black')

# 理论分布：对数正态
x = np.linspace(final_prices.min(), final_prices.max(), 200)
mu_theory = np.log(S0) + (r - 0.5 * sigma**2) * T
sigma_theory = sigma * np.sqrt(T)
pdf_theory = norm.pdf(np.log(x), mu_theory, sigma_theory) / x
axes[1].plot(x, pdf_theory, 'r-', linewidth=2, label='理论对数正态分布')

axes[1].axvline(x=S0 * np.exp(r * T), color='green', linestyle='--', 
                label=f'期望价格 {S0 * np.exp(r * T):.2f}')

axes[1].set_xlabel('到期价格', fontsize=12)
axes[1].set_ylabel('概率密度', fontsize=12)
axes[1].set_title('到期价格分布', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - GBM 生成的路径呈现指数增长特征（对数正态分布）")
print("  - 到期价格的分布是右偏的（有长尾）")
print("  - 理论期望价格 S0·e^(rT) = {:.2f}".format(S0 * np.exp(r * T)))

---
## 3. 蒙特卡洛定价欧式期权

### 3.1 定价公式

欧式看涨期权的 MC 定价：

$$\hat{C} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K, 0)$$

欧式看跌期权的 MC 定价：

$$\hat{P} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \max(K - S_T^{(i)}, 0)$$

### 3.2 置信区间

MC 估计的 95% 置信区间：

$$\hat{C} \pm 1.96 \cdot \frac{\sigma_{payoff}}{\sqrt{N}}$$

In [ ]:
def mc_european_option(S0, K, r, sigma, T, N, M=252, option_type='call'):
    """
    蒙特卡洛定价欧式期权
    
    Parameters
    ----------
    S0 : float - 初始价格
    K : float - 行权价
    r : float - 无风险利率
    sigma : float - 波动率
    T : float - 到期时间
    N : int - 模拟路径数量
    M : int - 时间步数（默认 252）
    option_type : str - 'call' 或 'put'
    
    Returns
    -------
    dict : 包含价格、标准误差、置信区间
    """
    # 模拟路径
    paths = simulate_gbm_paths(S0, r, sigma, T, M, N)
    final_prices = paths[:, -1]
    
    # 计算 payoff
    if option_type == 'call':
        payoffs = np.maximum(final_prices - K, 0)
    elif option_type == 'put':
        payoffs = np.maximum(K - final_prices, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    
    # 计算价格
    discount = np.exp(-r * T)
    price = discount * payoffs.mean()
    
    # 计算标准误差
    std_error = discount * payoffs.std() / np.sqrt(N)
    
    # 95% 置信区间
    ci_lower = price - 1.96 * std_error
    ci_upper = price + 1.96 * std_error
    
    return {
        'price': price,
        'std_error': std_error,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'payoffs': payoffs
    }


# BS 解析解（用于对比）
def bs_call(S, K, r, sigma, T):
    """BS 看涨期权公式"""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_put(S, K, r, sigma, T):
    """BS 看跌期权公式"""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


# 期权参数
K = 100  # 行权价（平值期权）

# MC 定价
np.random.seed(42)
mc_result = mc_european_option(S0, K, r, sigma, T, N=100000, option_type='call')
bs_price = bs_call(S0, K, r, sigma, T)

print(f"期权定价对比：")
print(f"  行权价 K = {K}")
print(f"  期权类型 = Call（看涨）")
print(f"\n  BS 解析解: {bs_price:.4f}")
print(f"  MC 模拟价: {mc_result['price']:.4f}")
print(f"  标准误差:  {mc_result['std_error']:.4f}")
print(f"  95% CI:    [{mc_result['ci_lower']:.4f}, {mc_result['ci_upper']:.4f}]")
print(f"\n  差异: {abs(mc_result['price'] - bs_price):.4f}")
print(f"  BS 价格是否在 CI 内: {'✓ 是' if mc_result['ci_lower'] <= bs_price <= mc_result['ci_upper'] else '✗ 否'}")

---
## 4. 方差缩减技术

### 4.1 为什么需要方差缩减？

MC 方法的主要问题是**方差大、收敛慢**。

标准误差 $SE = \sigma / \sqrt{N}$ 意味着：
- 精度提高 10 倍 → 需要 100 倍模拟次数
- 计算成本很高

**方差缩减技术**可以在不增加模拟次数的情况下，减小方差、提高精度。

### 4.2 对偶变量法 (Antithetic Variates)

**核心思想**：利用负相关性来减小方差。

如果 $Z$ 是标准正态随机变量，那么 $-Z$ 也是。对于每条路径，我们同时模拟：
- 路径 A：使用 $Z$
- 路径 B：使用 $-Z$

最终估计：

$$\hat{C} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N/2} \frac{f(S_T^{A,i}) + f(S_T^{B,i})}{2}$$

**为什么有效？**
- 如果 $Z$ 导致价格上升，$-Z$ 会导致价格下降
- 两者平均后，方差减小
- 期望值不变（无偏估计）

In [ ]:
def mc_european_antithetic(S0, K, r, sigma, T, N, M=252, option_type='call'):
    """
    蒙特卡洛定价（使用对偶变量法）
    
    Parameters
    ----------
    N : int - 总路径数量（必须是偶数）
    """
    if N % 2 != 0:
        N = N + 1  # 确保是偶数
    
    dt = T / M
    half_N = N // 2
    
    # 生成一半的随机数
    Z = np.random.standard_normal((half_N, M))
    
    # 正向路径
    paths_pos = np.zeros((half_N, M + 1))
    paths_pos[:, 0] = S0
    
    # 反向路径（对偶）
    paths_neg = np.zeros((half_N, M + 1))
    paths_neg[:, 0] = S0
    
    for t in range(M):
        drift = (r - 0.5 * sigma**2) * dt
        vol = sigma * np.sqrt(dt)
        
        paths_pos[:, t + 1] = paths_pos[:, t] * np.exp(drift + vol * Z[:, t])
        paths_neg[:, t + 1] = paths_neg[:, t] * np.exp(drift - vol * Z[:, t])  # 注意符号
    
    # 计算 payoff
    if option_type == 'call':
        payoffs_pos = np.maximum(paths_pos[:, -1] - K, 0)
        payoffs_neg = np.maximum(paths_neg[:, -1] - K, 0)
    else:
        payoffs_pos = np.maximum(K - paths_pos[:, -1], 0)
        payoffs_neg = np.maximum(K - paths_neg[:, -1], 0)
    
    # 对偶变量法：取平均
    payoffs_avg = (payoffs_pos + payoffs_neg) / 2
    
    discount = np.exp(-r * T)
    price = discount * payoffs_avg.mean()
    std_error = discount * payoffs_avg.std() / np.sqrt(half_N)
    
    return {
        'price': price,
        'std_error': std_error,
        'ci_lower': price - 1.96 * std_error,
        'ci_upper': price + 1.96 * std_error
    }


# 对比标准 MC 和对偶变量法
np.random.seed(42)
N_compare = 100000

# 标准 MC
mc_standard = mc_european_option(S0, K, r, sigma, T, N_compare, option_type='call')

# 对偶变量法
np.random.seed(42)
mc_antithetic = mc_european_antithetic(S0, K, r, sigma, T, N_compare, option_type='call')

print(f"方差缩减效果对比（N = {N_compare:,}）：\n")
print(f"{'方法':<15} {'价格':>10} {'标准误差':>10} {'方差缩减':>10}")
print("-" * 50)
print(f"{'BS 解析解':<15} {bs_price:>10.4f} {'--':>10} {'--':>10}")
print(f"{'标准 MC':<15} {mc_standard['price']:>10.4f} {mc_standard['std_error']:>10.4f} {'基准':>10}")
print(f"{'对偶变量法':<15} {mc_antithetic['price']:>10.4f} {mc_antithetic['std_error']:>10.4f} ", end="")

variance_ratio = (mc_standard['std_error'] / mc_antithetic['std_error'])**2
print(f"{variance_ratio:>9.2f}x")

print(f"\n解读：")
print(f"  - 对偶变量法的标准误差更小")
print(f"  - 方差缩减了约 {variance_ratio:.1f} 倍")
print(f"  - 等价于用 {(variance_ratio * N_compare):,.0f} 条标准 MC 路径达到相同精度")

### 4.3 控制变量法 (Control Variates)

**核心思想**：利用已知解析解的相似产品来减小方差。

假设我们要定价期权 A（未知），但有一个相似的期权 B（已知解析解）。

控制变量估计：

$$\hat{C}_A = \bar{C}_A - \beta (\bar{C}_B - C_B^{true})$$

其中 $\beta$ 是最优系数：

$$\beta = \frac{Cov(C_A, C_B)}{Var(C_B)}$$

**常用控制变量**：
- 用几何平均亚式期权（有解析解）控制算术平均亚式期权
- 用 BS Call 价格控制其他路径依赖期权

In [ ]:
def mc_european_control_variate(S0, K, r, sigma, T, N, M=252, option_type='call'):
    """
    蒙特卡洛定价（使用控制变量法）
    
    控制变量：使用 BS 解析解作为控制
    """
    # 模拟路径
    paths = simulate_gbm_paths(S0, r, sigma, T, M, N)
    final_prices = paths[:, -1]
    
    # 计算 payoff
    if option_type == 'call':
        payoffs = np.maximum(final_prices - K, 0)
        bs_true = bs_call(S0, K, r, sigma, T)
    else:
        payoffs = np.maximum(K - final_prices, 0)
        bs_true = bs_put(S0, K, r, sigma, T)
    
    # BS 的「MC 估计」：用同一条路径计算
    # 这里我们用 BS 公式的「模拟值」作为控制
    # 实际上，我们可以用任何已知量
    
    # 计算最优 beta
    discount = np.exp(-r * T)
    discounted_payoffs = discount * payoffs
    
    # 用标的价格作为控制变量（其期望已知：S0）
    control = final_prices
    control_mean = S0 * np.exp(r * T)  # 风险中性期望
    
    # 最优 beta
    cov_matrix = np.cov(discounted_payoffs, control)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]
    
    # 控制变量估计
    controlled_payoffs = discounted_payoffs - beta * (control - control_mean)
    price = controlled_payoffs.mean()
    std_error = controlled_payoffs.std() / np.sqrt(N)
    
    return {
        'price': price,
        'std_error': std_error,
        'ci_lower': price - 1.96 * std_error,
        'ci_upper': price + 1.96 * std_error,
        'beta': beta
    }


# 对比三种方法
np.random.seed(42)
mc_control = mc_european_control_variate(S0, K, r, sigma, T, N_compare, option_type='call')

print(f"三种方法对比（N = {N_compare:,}）：\n")
print(f"{'方法':<15} {'价格':>10} {'标准误差':>10} {'相对效率':>10}")
print("-" * 50)
print(f"{'BS 解析解':<15} {bs_price:>10.4f} {'--':>10} {'--':>10}")
print(f"{'标准 MC':<15} {mc_standard['price']:>10.4f} {mc_standard['std_error']:>10.4f} {'基准':>10}")

eff_antithetic = (mc_standard['std_error'] / mc_antithetic['std_error'])**2
print(f"{'对偶变量法':<15} {mc_antithetic['price']:>10.4f} {mc_antithetic['std_error']:>10.4f} {eff_antithetic:>9.2f}x")

eff_control = (mc_standard['std_error'] / mc_control['std_error'])**2
print(f"{'控制变量法':<15} {mc_control['price']:>10.4f} {mc_control['std_error']:>10.4f} {eff_control:>9.2f}x")

---
## 5. 收敛性分析

### 5.1 标准误差与 $\sqrt{n}$ 的关系

MC 估计的标准误差：

$$SE = \frac{\sigma}{\sqrt{N}}$$

这意味着：
- $N$ 增加 4 倍 → $SE$ 减半
- $N$ 增加 100 倍 → $SE$ 减为 1/10

### 5.2 绘制收敛曲线

In [ ]:
# 收敛性分析
N_values = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000]

# 存储结果
mc_prices = []
mc_errors = []
antithetic_prices = []
antithetic_errors = []

for N in N_values:
    # 标准 MC
    np.random.seed(42)
    result = mc_european_option(S0, K, r, sigma, T, N, option_type='call')
    mc_prices.append(result['price'])
    mc_errors.append(result['std_error'])
    
    # 对偶变量法
    np.random.seed(42)
    result = mc_european_antithetic(S0, K, r, sigma, T, N, option_type='call')
    antithetic_prices.append(result['price'])
    antithetic_errors.append(result['std_error'])

# 绘制收敛曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：价格收敛
axes[0].plot(N_values, mc_prices, 'b-o', label='标准 MC', markersize=6)
axes[0].plot(N_values, antithetic_prices, 'r-s', label='对偶变量法', markersize=6)
axes[0].axhline(y=bs_price, color='green', linestyle='--', linewidth=2, label=f'BS 解析解 ({bs_price:.4f})')

axes[0].set_xscale('log')
axes[0].set_xlabel('模拟路径数 N', fontsize=12)
axes[0].set_ylabel('期权价格', fontsize=12)
axes[0].set_title('价格收敛曲线', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右图：标准误差
axes[1].plot(N_values, mc_errors, 'b-o', label='标准 MC', markersize=6)
axes[1].plot(N_values, antithetic_errors, 'r-s', label='对偶变量法', markersize=6)

# 理论 1/√N 曲线
N_theory = np.array(N_values)
se_theory = mc_errors[0] * np.sqrt(N_values[0] / N_theory)
axes[1].plot(N_values, se_theory, 'g--', label='理论 1/√N', linewidth=2)

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('模拟路径数 N', fontsize=12)
axes[1].set_ylabel('标准误差 (log scale)', fontsize=12)
axes[1].set_title('标准误差收敛（对数坐标）', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n收敛性分析：")
print("  - 价格随 N 增加而收敛到 BS 解析解")
print("  - 标准误差按 1/√N 递减")
print("  - 对偶变量法始终比标准 MC 更稳定")

In [ ]:
# 计算方差缩减效率
print(f"方差缩减效率汇总：\n")
print(f"{'N':>10} {'标准MC SE':>12} {'对偶SE':>12} {'效率提升':>10}")
print("-" * 50)

for i, N in enumerate(N_values):
    efficiency = (mc_errors[i] / antithetic_errors[i])**2
    print(f"{N:>10,} {mc_errors[i]:>12.4f} {antithetic_errors[i]:>12.4f} {efficiency:>9.2f}x")

print(f"\n结论：")
print(f"  - 对偶变量法平均提升效率约 2-4 倍")
print(f"  - 等价于用 2-4 倍的模拟次数达到相同精度")
print(f"  - 实现简单，几乎没有额外计算成本")

---
## 6. 实战：不同行权价的期权定价

### 6.1 波动率微笑

让我们用 MC 方法定价不同行权价的期权，观察「波动率微笑」现象。

In [ ]:
# 不同行权价的期权定价
K_values = np.linspace(80, 120, 9)
N_vol = 100000

results = []
for K_strike in K_values:
    np.random.seed(42)
    mc_result = mc_european_antithetic(S0, K_strike, r, sigma, T, N_vol, option_type='call')
    bs_result = bs_call(S0, K_strike, r, sigma, T)
    
    # 计算隐含波动率（假设市场价 = BS 价格）
    # 这里我们用 BS 价格作为「市场价」
    
    results.append({
        'K': K_strike,
        'mc_price': mc_result['price'],
        'bs_price': bs_result,
        'moneyness': S0 / K_strike  # moneyness
    })

# 绘制价格对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

K_vals = [r['K'] for r in results]
mc_prices = [r['mc_price'] for r in results]
bs_prices = [r['bs_price'] for r in results]
moneyness = [r['moneyness'] for r in results]

# 左图：价格 vs 行权价
axes[0].plot(K_vals, mc_prices, 'b-o', label='MC 价格', markersize=6)
axes[0].plot(K_vals, bs_prices, 'r--s', label='BS 价格', markersize=6)
axes[0].axvline(x=S0, color='gray', linestyle=':', alpha=0.5, label=f'当前价格 S0={S0}')

axes[0].set_xlabel('行权价 K', fontsize=12)
axes[0].set_ylabel('期权价格', fontsize=12)
axes[0].set_title('不同行权价的 Call 价格', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右图：价格差异
price_diff = np.array(mc_prices) - np.array(bs_prices)
axes[1].bar(K_vals, price_diff, width=1.5, alpha=0.7, color='skyblue', edgecolor='black')
axes[1].axhline(y=0, color='black', linewidth=0.5)

axes[1].set_xlabel('行权价 K', fontsize=12)
axes[1].set_ylabel('MC - BS', fontsize=12)
axes[1].set_title('MC 与 BS 的差异', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - MC 价格与 BS 价格高度一致")
print("  - 差异来自随机噪声，增加 N 可以减小")
print("  - 深度虚值/实值期权的 MC 估计方差更大")

---
## 7. MC 方法的优缺点

### 7.1 优点

| 优点 | 说明 |
|------|------|
| **灵活** | 可以处理任意 payoff 结构 |
| **路径依赖** | 天然适合路径依赖期权（亚式、障碍等） |
| **高维** | 可以处理多个标的资产 |
| **直观** | 容易理解和实现 |

### 7.2 缺点

| 缺点 | 说明 |
|------|------|
| **慢** | 收敛速度 $O(1/\sqrt{N})$，精度提高 10 倍需要 100 倍计算 |
| **方差** | 估计有随机性，每次结果略有不同 |
| **希腊字母** | 计算 Greeks 需要额外技巧 |
| **美式期权** | 不能直接定价美式期权（需要 Longstaff-Schwartz 方法） |

### 7.3 何时使用 MC？

- **没有解析解**的复杂期权（亚式、障碍、篮子期权）
- **多资产**期权（相关性问题）
- **路径依赖**期权
- 作为**验证其他方法**的基准

---
## 8. 小结

### 核心收获

1. **蒙特卡洛方法**通过随机模拟来近似计算期权价格

2. **几何布朗运动**是模拟资产价格的标准模型：
   $$S_{t+\Delta t} = S_t \cdot \exp\left((r - \frac{\sigma^2}{2})\Delta t + \sigma\sqrt{\Delta t} \cdot Z\right)$$

3. **标准误差**与 $\sqrt{N}$ 成反比：
   - 精度提高 10 倍 → 计算量增加 100 倍
   - 这是 MC 方法的主要局限

4. **方差缩减技术**可以显著提高效率：
   - **对偶变量法**：利用负相关性，效率提升 2-4 倍
   - **控制变量法**：利用已知解析解，效率提升更多

5. MC 方法**灵活但慢**，适合复杂期权和多资产问题

### 关键公式

- GBM 离散化：$S_{t+\Delta t} = S_t \cdot \exp((r - \sigma^2/2)\Delta t + \sigma\sqrt{\Delta t} \cdot Z)$
- MC 定价：$\hat{C} = e^{-rT} \cdot \frac{1}{N} \sum \max(S_T - K, 0)$
- 标准误差：$SE = \sigma_{payoff} / \sqrt{N}$
- 对偶变量：$\hat{C} = e^{-rT} \cdot \frac{1}{N/2} \sum (f(S_T^+) + f(S_T^-)) / 2$

### 延伸阅读

- [Monte Carlo Methods in Financial Engineering](https://www.amazon.com/Monte-Carlo-Methods-Financial-Engineering/dp/0387004513) - Paul Glasserman
- [Monte Carlo simulation](https://en.wikipedia.org/wiki/Monte_Carlo_method) - Wikipedia

---
## 验收标准 Checklist

完成本次学习后，你应该能够：

- [x] **能模拟资产价格路径**：我们实现了 `simulate_gbm_paths()` 函数
- [x] **理解标准误差与 √n 的关系**：
  - 标准误差 $SE = \sigma / \sqrt{N}$
  - 精度提高 10 倍需要 100 倍模拟次数
  - 绘制了收敛曲线验证了这一关系
- [x] **能用对偶变量做方差缩减**：
  - 实现了 `mc_european_antithetic()` 函数
  - 效率提升约 2-4 倍
  - 理解了负相关性减小方差的原理

### 自测题

1. 为什么 MC 方法的收敛速度是 $O(1/\sqrt{N})$？这与什么数学定理有关？
2. 对偶变量法为什么能减小方差？它的前提假设是什么？
3. 如果你想将 MC 估计的标准误差从 0.1 降到 0.01，需要增加多少倍的模拟次数？
4. MC 方法相比 BS 公式的优势是什么？劣势是什么？

---

**恭喜你完成了蒙特卡洛期权定价的学习！** 🎉

下次我们将学习**亚式期权定价**，了解 MC 方法如何处理路径依赖期权。